The Model

In [135]:
import pandas as pd 
import streamlit as st 
import numpy as np
import sklearn as sk
import seaborn as sns

In [136]:
df = pd.read_csv('/Users/duncanstangel/Documents/GitHub/STANGEL-Data-Science-Portfolio/sml_streamlit_app/congressional_voting_records.csv').dropna().fillna(0.5)
features = df[['handicapped-infants', 'water-project-cost-sharing', 'physician-fee-freeze', 
               'el-salvador-aid', 'religious-groups-in-schools', 'anti-satellite-test-ban',
               'aid-to-nicaraguan-contras', 'mx-missile', 'immigration', 'synfuels-corporation-cutback',
               'education-spending', 'superfund-right-to-sue', 'crime', 'duty-free-exports', 
               'export-administration-act-south-africa']] #laws can be more broadly generalized into buckets like "environmental_regulation" and "trade_deals", this allows for more generalizability for future datasets
X = features
Y = df['party'] 
st.dataframe(df)

2026-04-11 11:43:14.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 11:43:14.546 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 11:43:14.552 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [137]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score


vote_map = {
    'democrat' : 0.0,
    'republican' : 1.0}
feature_map = {
    'y' : 1.0,
    'n' : 0.0,
    '?' : 0.5}

X_numeric = X.replace(feature_map)
Y_numeric = Y.replace(vote_map)

X_train, X_test, Y_train, Y_test = train_test_split(X_numeric, Y_numeric,
                                                    test_size=0.2,
                                                    random_state=42)

model = DecisionTreeClassifier(random_state = 42, max_depth = 4)
model.fit(X_train, Y_train)
model.feature_importances_

/var/folders/9f/f2r2bq793nqd3zrv8mxk8plm0000gn/T/ipykernel_35644/2963879642.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_numeric = X.replace(feature_map)
/var/folders/9f/f2r2bq793nqd3zrv8mxk8plm0000gn/T/ipykernel_35644/2963879642.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Y_numeric = Y.replace(vote_map)


array([4.74022720e-03, 0.00000000e+00, 9.06531293e-01, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 8.80327908e-03,
       1.35544366e-02, 4.35163452e-02, 0.00000000e+00, 0.00000000e+00,
       5.96408168e-04, 3.93964928e-03, 1.83183618e-02])

In [138]:
import graphviz 
from sklearn import tree 

dot_data = tree.export_graphviz(model, feature_names = X.columns.tolist(), class_names=['democrat', 'republican'], filled=True)
graph = graphviz.Source(dot_data)
st.graphviz_chart(graph)

2026-04-11 11:43:14.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 11:43:14.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 11:43:14.602 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-11 11:43:14.603 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [139]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [2, 3, 4, 5],
    'min_samples_split': [2, 3, 4, 5, 6],
    'min_samples_leaf': [2, 3, 4, 5, 6],
    'class_weight' : [None, 'balanced']
} # allows for customization of hyperparameters to fine tune model 

grid_search = GridSearchCV(estimator = model,
                           param_grid = param_grid,
                           cv = 5,
                           scoring = 'recall',
                           verbose = 3)
grid_search.fit(X_train, Y_train) # allows for param_grid to be searched fpr optimal model 

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

Fitting 5 folds for each of 600 candidates, totalling 3000 fits
[CV 1/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=2;, score=1.000 total time=   0.0s
[CV 2/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=2;, score=0.964 total time=   0.0s
[CV 3/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=2;, score=0.926 total time=   0.0s
[CV 4/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=2;, score=0.963 total time=   0.0s
[CV 5/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=2;, score=1.000 total time=   0.0s
[CV 1/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=3;, score=1.000 total time=   0.0s
[CV 2/5] END class_weight=None, criterion=gini, max_depth=2, min_samples_leaf=2, min_samples_split=3;, score=0.964 total time=   0.0

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

best_model = grid_search.best_estimator_

Y_pred = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(Y_test, Y_pred))

In [ ]:
df_melted = df.melt(id_vars=['party'],
        value_vars=df.columns[1:], 
        var_name = 'issue',
        value_name='vote_value')

df_melted['vote_value'] = df_melted['vote_value'].map({'y': 1, 'n': 0})

visualization = plt.figure(figsize=(12,6))
sns.barplot(x = 'issue', y = 'vote_value', hue='party', data=df_melted)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
st.pyplot(visualization)

In [ ]:
cm = confusion_matrix(Y_test, Y_pred)
plt.figure(figsize=(8, 6))
heatmap = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
st.pyplot(heatmap.figure)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

best_model = grid_search.best_estimator_ #isolates best model 

Y_pred = best_model.predict(X_test) # creates predictive value of model 

print("Classification Report:")
print(classification_report(Y_test, Y_pred))

st.header("Input Features") ## allows for radio button voting
handicapped_infants = st.radio("How would you vote on ____?", ['y', 'n'])
water_project_cost_sharing = st.radio("How would you vote on _____?", ['y', 'n'])
adoption_of_the_budget_resolution = st.radio("How would you vote on _____?", ['y', 'n'])
physician_fee_freeze = st.radio("How would you vote on _____?", ['y', 'n'])
el_salvador_aid = st.radio("How would you vote on _____?", ['y', 'n'])
religious_groups_in_schools = st.radio("How would you vote on _____?", ['y', 'n'])
anti_satellite_test_ban = st.radio("How would you vote on _____?", ['y', 'n'])
aid_to_nicaraguan_contras = st.radio("How would you vote on _____?", ['y', 'n'])
mx_missile = st.radio("How would you vote on _____?", ['y', 'n'])
immigration = st.radio("How would you vote on _____?", ['y', 'n'])
synfuels_corporation_cutback = st.radio("How would you vote on _____?", ['y', 'n'])
education_spending = st.radio("How would you vote on _____?", ['y', 'n'])
superfund_right_to_sue = st.radio("How would you vote on _____?", ['y', 'n'])
crime = st.radio("How would you vote on _____?", ['y', 'n'])
duty_free_exports = st.radio("How would you vote on _____?", ['y', 'n'])
export_administration_act_south_africa = st.radio("How would you vote on _____?", ['y', 'n'])

input_data = {
    'handicapped-infants': 1 if 'handicapped-infants' == 'y' else 0,
    'water-project-cost-sharing': 1 if 'water-project-cost-sharing' == 'y' else 0,
    'adoption-of-the-budget-resolution': 1 if 'adoption-of-the-budget-resolution' == 'y' else 0,
    'physician-fee-freeze': 1 if 'physician-fee-freeze' == 'y' else 0,
    'el-salvador-aid': 1 if 'el-salvador-aid' == 'y' else 0,
    'religious-groups-in-schools': 1 if 'religious-groups-in-schools' == 'y' else 0,
    'anti-satellite-test-ban': 1 if 'anti-satellite-test-ban' == 'y' else 0,
    'aid-to-nicaraguan-contras': 1 if 'aid-to-nicaraguan-contras' == 'y' else 0,
    'mx-missile': 1 if 'mx-missile' == 'y' else 0,
    'immigration': 1 if 'immigration' == 'y' else 0,
    'synfuels-corporation-cutback': 1 if 'synfuels-corporation-cutback' == 'y' else 0,
    'education-spending': 1 if 'education-spending' == 'y' else 0,
    'superfund-right-to-sue': 1 if 'superfund-right-to-sue' == 'y' else 0,
    'crime': 1 if crime == 'y' else 0,
    'duty-free-exports': 1 if 'duty-free_-xports' == 'y' else 0,
    'export-administration-act-south-africa': 1 if 'export-administration-act-south-africa' == 'y' else 0
}

input_df = pd.DataFrame([input_data])
input_df = input_df[X_test.columns]

Y_pred_prob = model.predict_proba(input_df) # uses user input to generate result; replace X_test with something relating to radio buttons

df_pred_prob = pd.DataFrame([input_data]) ## Basic structure; needs complexity from addition of radio buttons 

st.subheader('Predicted Political Party')
st.dataframe(column_config={
               'Democrat': st.column_config.ProgressColumn(
                 'Democrat',
                 format='%f',
                 width='medium',
                 min_value=0,
                 max_value=1
               ),
               'Republican': st.column_config.ProgressColumn(
                 'Republican',
                 format='%f',
                 width='medium',
                 min_value=0,
                 max_value=1
               ),
             }, hide_index=True)